# 3. QUBO Solver basics

A short companion to [Solving a QUBO problem easily](https://docs.pasqal.com/applicationsolvingtools/qubo/tutorials/01-qubosolver-basics/), the third core tutorial in the pre-reading PDF. We solve the same tiny problem using a local quantum emulator and a classical solver.

Run `uv sync --locked` in the repository terminal, select `.venv/bin/python` as the kernel, and run the cells in order. We omit the tutorial's remote backends: no credentials, cloud jobs, or QPU access are needed.

In [1]:
from qubosolver import (
    ClassicalSolvingConfig,
    Instance,
    LocalEmulator,
    QuantumSolvingConfig,
    Solver,
    SolverConfig,
    analysis,
    matrix,
)

## Define the problem

A QUBO minimizes $x^T Q x$ over binary variables. Diagonal entries reward or penalize selecting a variable; off-diagonal entries couple pairs. In this symmetric matrix, each off-diagonal pair contributes twice in $x^T Q x$.

The negative diagonal entries favor selecting variables, while the positive couplings discourage selecting the third variable together with either of the first two.

In [2]:
Q = matrix.tensor([
    [-0.2, 0.0, 1.0],
    [0.0, -1.0, 1.5],
    [1.0, 1.5, -0.1],
])
instance = Instance(Q)

## Solve with a local quantum emulator

`Instance` holds the problem, `SolverConfig` chooses the approach, and `Solver.solve()` runs it. The quantum workflow handles preprocessing, atom placement, pulse design, local emulation, and postprocessing with default settings. This is a simulation, not execution on quantum hardware.

In [3]:
quantum_config = SolverConfig(
    solving=QuantumSolvingConfig(backend=LocalEmulator())
)
quantum_solution = Solver(instance, quantum_config).solve()
quantum_results = analysis.to_dataframe([quantum_solution])
quantum_results.sort_values("costs")

,labels,bitstrings,costs,counts,probs
0,0,110,-1.2,986,0.986
1,0,001,-0.1,14,0.014


## Solve with a classical baseline

Tabu search explores binary candidates on the CPU. We use the same instance and give it a short time budget. Both approaches are heuristic in general; neither guarantees the optimum for arbitrary problems.

In [4]:
classical_config = SolverConfig(
    solving=ClassicalSolvingConfig(algorithm="tabu_search", time_limit=1.0)
)
classical_solution = Solver(instance, classical_config).solve()
classical_results = analysis.to_dataframe([classical_solution])
classical_results.sort_values("costs")

,labels,bitstrings,costs,counts,probs
0,0,110,-1.2,1,1.0


## Read the results

`bitstrings` are candidate assignments and `costs` are their objective values: lower is better. For quantum samples, `counts` and `probs` describe their frequency. Defaults include classical postprocessing, so the reported solutions are not necessarily raw emulator measurements.

For this three-variable problem, `110` has the optimal cost of -1.2: it selects the first two variables without paying a pairwise penalty. Sample frequencies can vary between runs.

In [5]:
print(f"Best quantum-workflow cost: {quantum_results['costs'].min():.3f}")
print(f"Best classical cost:       {classical_results['costs'].min():.3f}")

Best quantum-workflow cost: -1.200
Best classical cost:       -1.200
